In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("final_healthcare_master.csv")

df.head()

,ranking,district,state,population,growth,sex_ratio,literacy,health_centres,population_per_health_centre,healthcare_density,healthcare_rank,electricity,clean_fuel,health_insurance,female_literacy,antenatal_care,institutional_birth,vaccination,stunted,underweight,anaemia
0,1,THANE,MAHARASHTRA,11060148.0,36.01 %,886,84.53,336.0,3.291711e+04,Poor,134.0,99.29,90.40,22.73,90.49,70.19,93.62,-74.9,40.76,30.81,58.82
1,2,NORTH TWENTY FOUR PARGANAS,WEST BENGAL,10009781.0,12.04 %,955,84.06,870.0,1.150550e+04,Average,7.0,98.50,48.04,34.60,75.73,60.25,92.16,NaN,32.31,29.12,56.94
2,3,BANGALORE,KARNATAKA,9621551.0,47.18 %,916,87.67,0.0,9.621551e+06,Critical,361.0,98.78,97.20,28.83,87.27,74.60,99.34,-78.16,31.26,28.09,35.54
3,4,PUNE,MAHARASHTRA,9429408.0,30.37 %,915,86.15,773.0,1.219846e+04,Average,16.0,99.37,89.49,14.42,89.01,68.59,97.97,-58.08,30.70,32.66,51.93
4,5,MUMBAI SUBURBAN,MAHARASHTRA,9356962.0,8.29 %,860,89.91,0.0,9.356962e+06,Critical,361.0,99.17,98.68,20.19,91.58,72.20,98.08,*,37.23,24.59,49.97


In [ ]:
features = [

    "population_per_health_centre",

    "vaccination",

    "anaemia",

    "stunted",

    "underweight",

    "female_literacy",

    "institutional_birth",

    "antenatal_care"

]

In [ ]:
def clean_numeric_column(df, column):

    df[column] = (

        df[column]

        .astype(str)

        .str.replace("*","",regex=False)

        .str.replace(",","",regex=False)

        .str.replace("--","",regex=False)

        .str.replace("NA","",regex=False)

        .str.replace("N/A","",regex=False)

        .str.strip()

    )

    df[column] = pd.to_numeric(

        df[column],

        errors="coerce"

    )

    df[column] = df[column].fillna(

        df[column].median()

    )

    return df

In [ ]:
for col in features:

    df = clean_numeric_column(df,col)

print(df[features].dtypes)

population_per_health_centre    float64
vaccination                     float64
anaemia                         float64
stunted                         float64
underweight                     float64
female_literacy                 float64
institutional_birth             float64
antenatal_care                  float64
dtype: object


In [ ]:
scaler = MinMaxScaler()

df_scaled = df.copy()

df_scaled[features] = scaler.fit_transform(df[features])

In [ ]:
good_columns = [

    "vaccination",

    "female_literacy",

    "institutional_birth",

    "antenatal_care"

]

for col in good_columns:

    df_scaled[col] = 1 - df_scaled[col]

In [ ]:
weights = {

    "population_per_health_centre":0.30,

    "vaccination":0.20,

    "anaemia":0.15,

    "stunted":0.10,

    "underweight":0.10,

    "female_literacy":0.05,

    "institutional_birth":0.05,

    "antenatal_care":0.05

}

df_scaled["healthcare_priority_score"] = (

      weights["population_per_health_centre"] * df_scaled["population_per_health_centre"]

    + weights["vaccination"] * df_scaled["vaccination"]

    + weights["anaemia"] * df_scaled["anaemia"]

    + weights["stunted"] * df_scaled["stunted"]

    + weights["underweight"] * df_scaled["underweight"]

    + weights["female_literacy"] * df_scaled["female_literacy"]

    + weights["institutional_birth"] * df_scaled["institutional_birth"]

    + weights["antenatal_care"] * df_scaled["antenatal_care"]

) * 100

In [ ]:
conditions = [

    df_scaled["healthcare_priority_score"]>=75,

    df_scaled["healthcare_priority_score"]>=60,

    df_scaled["healthcare_priority_score"]>=40

]

choices = [

    "Critical",

    "High",

    "Medium"

]

df_scaled["priority_level"] = np.select(

    conditions,

    choices,

    default="Low"

)

In [ ]:
df_scaled["priority_rank"] = (

    df_scaled["healthcare_priority_score"]

    .rank(

        ascending=False,

        method="dense"

    )

    .astype(int)

)

In [ ]:
top_priority = (

df_scaled

.sort_values(

"healthcare_priority_score",

ascending=False

)

[[

"district",

"state",

"healthcare_priority_score",

"priority_level",

"priority_rank"

]]

.head(25)

)

top_priority

,district,state,healthcare_priority_score,priority_level,priority_rank
2,BANGALORE,KARNATAKA,66.640940,High,1
7,AHMADABAD,GUJARAT,65.716041,High,2
43,VISAKHAPATNAM,ANDHRA PRADESH,57.427235,Medium,3
34,KOLKATA,WEST BENGAL,56.207340,Medium,4
4,MUMBAI SUBURBAN,MAHARASHTRA,54.229173,Medium,5
453,TAPI,GUJARAT,52.574554,Medium,6
146,TIRUVANNAMALAI,TAMIL NADU,50.077368,Medium,7
540,DEBAGARH,ORISSA,49.861431,Medium,8
290,BHARUCH,GUJARAT,49.579727,Medium,9
79,BANKURA,WEST BENGAL,49.183023,Medium,10


In [ ]:
import plotly.express as px

fig = px.bar(

top_priority,

x="district",

y="healthcare_priority_score",

color="priority_level",

title="Top 25 Healthcare Priority Districts"

)

fig.show()

In [ ]:
fig = px.histogram(

df_scaled,

x="priority_level",

color="priority_level",

title="Healthcare Priority Distribution"

)

fig.show()

In [ ]:
df_scaled.to_csv(

"healthcare_priority_index.csv",

index=False

)



In [ ]:
df_scaled[[

"district",

"state",

"healthcare_priority_score",

"priority_level",

"priority_rank"

]].head(20)

,district,state,healthcare_priority_score,priority_level,priority_rank
0,THANE,MAHARASHTRA,43.143000,Medium,72
1,NORTH TWENTY FOUR PARGANAS,WEST BENGAL,28.488685,Low,430
2,BANGALORE,KARNATAKA,66.640940,High,1
3,PUNE,MAHARASHTRA,38.714485,Low,161
4,MUMBAI SUBURBAN,MAHARASHTRA,54.229173,Medium,5
5,SOUTH TWENTY FOUR PARGANAS,WEST BENGAL,28.474635,Low,437
6,BARDDHAMAN,WEST BENGAL,28.479487,Low,433
7,AHMADABAD,GUJARAT,65.716041,High,2
8,MURSHIDABAD,WEST BENGAL,32.889775,Low,298
9,JAIPUR,RAJASTHAN,24.065228,Low,561
